# BarGPT v2 inspection

This notebook is an executable review surface for the model contract. It writes generated artifacts under `D:/TradingML/runtimes/bar_gpt/v2/inspection`, never into the repository. Set `CHECKPOINT` to evaluate a saved model.

In [ ]:
from pathlib import Path
import json, sys, torch
REPO = Path(r'D:/TradingCodes/quant-research-workbench')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
OUT = Path(r'D:/TradingML/runtimes/bar_gpt/v2/inspection'); OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = Path(r'')  # set to checkpoint_latest.pt to evaluate
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, 'torch:', torch.__version__)

## Architecture, torchinfo summary, and torchview graph

In [ ]:
from research.bar_gpt.v2.config import BarGPTConfig, DataConfig
from research.bar_gpt.v2.data import TIMEFRAME_US_BY_NAME, PATHWAY_ID_BY_NAME
from research.bar_gpt.v2.model import BarGPTV2, build_model_mermaid
from research.mlops.model_artifacts import parameter_summary, write_model_artifacts
model_config, data_config = BarGPTConfig(), DataConfig()
model = BarGPTV2(model_config).to(DEVICE).eval()
view_names = tuple(TIMEFRAME_US_BY_NAME)
def dummy():
    views = {n: torch.zeros(1, 8, model_config.feature_dim, device=DEVICE) for n in view_names}
    asof = {n: torch.full((1, 1), 7, dtype=torch.long, device=DEVICE) for n in view_names}
    return (), {'views': views, 'timeframe_us': {n: TIMEFRAME_US_BY_NAME[n] for n in view_names}, 'pathway_ids': {n: PATHWAY_ID_BY_NAME[n] for n in view_names}, 'base_view': '1s', 'origin_indices': torch.zeros(1, 1, dtype=torch.long, device=DEVICE), 'asof_indices': asof, 'horizon_ids': torch.arange(len(data_config.horizons_us), device=DEVICE)}
write_model_artifacts(model=model, artifact_dir=OUT/'model', model_config=model_config, input_contract={'views': {n: ['B','T',model_config.feature_dim] for n in view_names}, 'asof_indices': ['B','N_views'], 'origin_indices': ['B','N_origins']}, output_contract={'embeddings': ['B','N_origins',model_config.d_model], 'horizon_quantiles': ['B','N_origins','H',model_config.target_dim-8,len(model_config.quantiles)], 'horizon_return_class_logits': ['B','N_origins','H',12,5]}, architecture_mermaid=build_model_mermaid(), summary_notes='Interactive BarGPT v2 inspection.', dummy_input_factory=dummy)
print(parameter_summary(model))
artifact_dir = OUT/'model'
print('artifacts:', [p.name for p in sorted(artifact_dir.iterdir())])
for name in ('model_summary_torchinfo.txt', 'model_summary.txt', 'model_summary_torchinfo_error.txt'):
    path = artifact_dir/name
    if path.exists():
        print(f'\n--- {name} ---\n{path.read_text()[:4000]}')
        break

## Batch and target shape inspection

In [ ]:
from research.bar_gpt.v2.train import _dummy_example
from research.bar_gpt.v2.data import collate_examples
batch = collate_examples([_dummy_example(data_config)])
print('origins:', batch.origin_count, 'views:', {k: tuple(v.shape) for k,v in batch.views.items()})
print('as-of:', {k: tuple(v.shape) for k,v in batch.asof_indices.items()})
print('horizon_targets:', None if batch.horizon_targets is None else tuple(batch.horizon_targets.shape))
print('horizon_mask valid:', None if batch.horizon_mask is None else int(batch.horizon_mask.sum()))
print('autoregressive targets:', {k: tuple(v.shape) for k,v in batch.autoregressive_targets.items()})
print('autoregressive masks:', {k: int(v.sum()) for k,v in batch.autoregressive_mask.items()})

## Checkpoint evaluation on a bounded validation sample

In [ ]:
from research.bar_gpt.v2 import assert_checkpoint_version
from research.bar_gpt.v2.train import _forward, validate
from research.bar_gpt.v2.config import ExperimentConfig, TrainConfig
from torch.utils.data import DataLoader
if CHECKPOINT:
    payload = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
    assert_checkpoint_version(payload)
    model.load_state_dict(payload['model'], strict=True)
    eval_config = ExperimentConfig(model=model_config, data=data_config, train=TrainConfig(validation_batches=4, cuda_prefetch=False, amp=False))
    loader = DataLoader([batch], batch_size=None)
    print('checkpoint samples:', payload.get('samples_seen'), 'optimizer steps:', payload.get('optimizer_steps'))
    print(validate(model, loader, eval_config, DEVICE))
else:
    print('Set CHECKPOINT to evaluate a saved checkpoint.')